In [ ]:
import os
import sys

import matplotlib.pyplot as plt
import tensorflow as tf

sys.path.append(os.path.abspath("../"))
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"

import glob

from train.models import FullModel
from util import dataset as u_dataset
from util import image as u_image

In [ ]:
batch_size = 32

encoder_name = "20250915-105208.keras"
classifier_ball_name = "20250915-105208.keras"
classifier_penaltyMark_name = "20250915-105208.keras"

path_to_models = "../../models"

#### Load Test Data


In [ ]:
path_to_test_data = glob.glob("../../data/test_ds*.tfrecords")

test_ds = u_dataset.get_dataset(path_to_test_data)
test_ds = test_ds.batch(batch_size, drop_remainder=False)

#### Load Model

In [ ]:
model = FullModel.load(
    (480, 320), filepath=path_to_models, filename=encoder_name, encoder_only=False, verbose=True
)
model.compile(optimizer=tf.keras.optimizers.Adam(), jit_compile=False)

#### Predict Data

In [ ]:
groundtruth_dataset = []
for x in test_ds:
    groundtruth_dataset.append(x)

# Take only image, camera, intrinsics for input
input_data = test_ds.map(lambda x: (x["image"], x["camera"], x["intrinsics"]))
# Convert tf.Data.Dataset into list
input_list = []
for x, y, z in input_data:
    input_list.append((x, y, z))

model.run_eagerly = True
# predictions = [model.predict(x=batch) for batch in input_list]

#### Evaluate Models

In [ ]:
results = [model.evaluate(x=batch) for batch in groundtruth_dataset]

# Reduce mean over batch axis
results_mean = tf.reduce_mean(results, axis=0)

In [ ]:
tf.print("Classifier BCE: ", results_mean[0])
tf.print("Classifier MSE: ", results_mean[2])
tf.print("Classifier Loss: ", results_mean[1])
tf.print("Encoder BCE: ", results_mean[3])
tf.print("Encoder MSE: ", results_mean[5])
tf.print("Encoder Loss: ", results_mean[4])
tf.print("Total Loss: ", results_mean[6])

#### Accuracy, Precision, Recall


In [ ]:
# # precision_metric = tf.keras.metrics.Precision()
# # recall_metric = tf.keras.metrics.Recall()
# # accuracy_metric = tf.keras.metrics.BinaryAccuracy()

# # mae_metric = tf.keras.metrics.MeanAbsoluteError()
# mae_metric_ball = u_metrics.MAE(err_threshold=0.2, object_name="ball")
# mae_metric_penaltyMark = u_metrics.MAE(err_threshold=0.2, object_name="penaltyMark")

# rmse_metric_ball = u_metrics.RMSE(err_threshold=0.2, object_name="ball")
# rmse_metric_penaltyMark = u_metrics.RMSE(err_threshold=0.2, object_name="penaltyMark")

# for test_data, groundtruth_data in zip(test_dataset, groundtruth_dataset, strict=True):
#     mae_metric_ball.update_state(groundtruth_data, test_data)
#     mae_metric_penaltyMark.update_state(groundtruth_data, test_data)

#     rmse_metric_ball.update_state(groundtruth_data, test_data)
#     rmse_metric_penaltyMark.update_state(groundtruth_data, test_data)


# mae_ball = mae_metric_ball.result().numpy()
# mae_metric_penaltyMark = mae_metric_penaltyMark.result().numpy()

# rmse_ball = rmse_metric_ball.result().numpy()
# rmse_metric_penaltyMark = rmse_metric_penaltyMark.result().numpy()

# print("MAE Ball: ", mae_ball)
# print("MAE PenaltyMark: ", mae_metric_penaltyMark)
# print("RMSE Ball: ", rmse_ball)
# print("RMSE PenaltyMark: ", rmse_metric_penaltyMark)

In [ ]:
image, camera, intrinsics = input_list[0]

In [ ]:
# Convert image from yuyv to yuv
image_yuv_stack = tf.stack(
    [
        image[..., 0],
        image[..., 1],
        image[..., 3],
        image[..., 2],
        image[..., 1],
        image[..., 3],
    ],
    axis=-1,
)
image_yuv = tf.reshape(
    image_yuv_stack, (tf.shape(image)[0], tf.shape(image)[1], tf.shape(image)[2] * 2, 3)
)  # [B, H_in, W_in*2, 3]

#### Draw example image

In [ ]:
plt.imshow(image_yuv[0][..., 0], cmap="gray")
plt.title("Raw Example Image in Grayscale")
plt.show()

#### Show Groundtruth Masks on Image


In [ ]:
# u_image.show_masks_on_image(test_directory, label, "ball", mask_name=None, grid_dims=(15, 20))
# u_image.show_masks_on_image(
#     test_directory, label, "ball", mask_name="objectness", grid_dims=(15, 20)
# )
# u_image.show_masks_on_image(test_directory, label, "ball", mask_name="loss", grid_dims=(15, 20))

#### Show Predicted Patches


In [ ]:
object_name = "penaltyMark"
sample_index = 10

# Extract a single sample at the sample_index
sample = u_dataset.get_sample_at_index(groundtruth_dataset[0], sample_index, keep_batch=True)

In [ ]:

results = model(
    (
        tf.expand_dims(image[sample_index], axis=0),
        tf.expand_dims(camera[sample_index], axis=0),
        tf.expand_dims(intrinsics[sample_index], axis=0),
    ),
    training=None,
)

model.evaluate(x=sample)

u_image.show_patches_on_image(image_yuv[sample_index], object_name, results)

tf.print("Candidate 1: ", "Probablity: ", results[object_name]["classification"][0][0], " Masks: ", results[object_name]["masks"][0][0])
tf.print("Candidate 2: ", "Probablity: ", results[object_name]["classification"][0][1], " Masks: ", results[object_name]["masks"][0][1])
tf.print("Candidate 3: ", "Probablity: ", results[object_name]["classification"][0][2], " Masks: ", results[object_name]["masks"][0][2])
tf.print("Candidate 4: ", "Probablity: ", results[object_name]["classification"][0][3], " Masks: ", results[object_name]["masks"][0][3])
tf.print("Candidate 5: ", "Probablity: ", results[object_name]["classification"][0][4], " Masks: ", results[object_name]["masks"][0][4])